<a href="https://colab.research.google.com/github/palakbhatt1/Android-Development-Assignment/blob/main/llm_prompting_and_synthetic_data_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq scikit-learn pandas numpy tqdm

In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('lab4_groq_apikey')  # Make sure this matches your secret name

if api_key is None:
    raise ValueError("API key not found. Add it in Colab Secrets.")

client = Groq(api_key=api_key)

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [ ]:
test_prompts = [
    "Write a Python function to check if a number is prime.",
    "Write a Python function to reverse a string.",
    "Write a Python function to compute factorial using recursion.",
    "Write a Python function to find the largest element in a list.",
    "Write a Python function to check palindrome.",
    "Write a Python function to sort a list without using built-in sort.",
    "Write a Python function to compute Fibonacci sequence.",
    "Write a Python function to remove duplicates from a list.",
    "Write a Python function to count vowels in a string.",
    "Write a Python function to merge two dictionaries.",
    "Write a Python function to check if a list is empty.",
    "Write a Python function to find GCD of two numbers.",
    "Write a Python function to flatten a nested list.",
    "Write a Python function to convert Celsius to Fahrenheit.",
    "Write a Python function to find second largest number.",
    "Write a Python function to calculate power without using pow.",
    "Write a Python function to find intersection of two lists.",
    "Write a Python function to check anagram.",
    "Write a Python function to find length of string without len().",
    "Write a Python function to generate multiplication table."
]

In [ ]:
import time

def call_llm(prompt):
    start = time.time()

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    end = time.time()

    latency = end - start
    tokens = response.usage.total_tokens
    output = response.choices[0].message.content.strip()

    return output, latency, tokens

In [ ]:
import json

# Check JSON format compliance (for structured prompt)
def check_json_valid(output):
    try:
        json.loads(output)
        return True
    except:
        return False

# Check consistency across 3 runs
def check_consistency(func, text, runs=3):
    outputs = []
    for _ in range(runs):
        out, _, _ = func(text)
        outputs.append(out)
    return len(set(outputs)) == 1

In [ ]:
def zero_shot(text):
    prompt = f"{text}"
    return call_llm(prompt)

In [ ]:
def role_based(text):
    prompt = f"""
    You are a professional Python developer.
    Generate clean, optimized Python code.

    {text}
    """
    return call_llm(prompt)

In [ ]:
def few_shot(text):
    prompt = f"""
    Generate Python code.

    Example 1:
    Instruction: Write a function to add two numbers.
    Code:
    def add(a, b):
        return a + b

    Example 2:
    Instruction: Write a function to check if number is even.
    Code:
    def is_even(n):
        return n % 2 == 0

    Now:
    Instruction: {text}
    Code:
    """
    return call_llm(prompt)

In [ ]:
def structured_prompt(text):
    prompt = f"""
    Generate Python code.

    Respond ONLY in valid JSON format:
    {{
        "function_name": "",
        "code": ""
    }}

    Instruction: {text}
    """
    return call_llm(prompt)

In [ ]:
import pandas as pd

strategies = {
    "Zero-shot": zero_shot,
    "Role-based": role_based,
    "Few-shot": few_shot,
    "Structured": structured_prompt
}

results = []

for strategy_name, func in strategies.items():
    for text in test_prompts:
        output, latency, tokens = func(text)

        # Format compliance (only meaningful for structured prompt)
        if strategy_name == "Structured":
            format_ok = check_json_valid(output)
        else:
            format_ok = None

        # Consistency check
        consistent = check_consistency(func, text)

        results.append({
            "Strategy": strategy_name,
            "Prompt": text,
            "Output": output,
            "Latency": latency,
            "Tokens": tokens,
            "Format_Compliance": format_ok,
            "Consistent": consistent
        })

df_results = pd.DataFrame(results)
df_results.head()

,Strategy,Prompt,Output,Latency,Tokens,Format_Compliance,Consistent
0,Zero-shot,Write a Python function to check if a number i...,**Prime Number Checker Function**\n===========...,0.542464,448,None,False
1,Zero-shot,Write a Python function to reverse a string.,## Reversing a String in Python\n\nYou can use...,0.433971,353,None,False
2,Zero-shot,Write a Python function to compute factorial u...,**Recursive Factorial Function in Python**\n==...,0.567077,447,None,False
3,Zero-shot,Write a Python function to find the largest el...,**Finding the Largest Element in a List**\n===...,1.477644,380,None,False
4,Zero-shot,Write a Python function to check palindrome.,**Palindrome Checker Function**\n=============...,2.359387,298,None,False


In [ ]:
df_results.groupby("Strategy")[["Latency", "Tokens"]].mean()

,Latency,Tokens
Strategy,,
Few-shot,2.480985,464.50
Role-based,2.572887,489.75
Structured,2.207388,151.55
Zero-shot,2.282765,433.55


In [ ]:
df_results.groupby("Strategy")["Consistent"].mean()

,Consistent
Strategy,
Few-shot,0.00
Role-based,0.00
Structured,0.05
Zero-shot,0.00


In [ ]:
df_results.groupby("Strategy")["Format_Compliance"].mean()

,Format_Compliance
Strategy,
Few-shot,NaN
Role-based,NaN
Structured,0.55
Zero-shot,NaN


In [ ]:
sample_prompt = test_prompts[0]

for i in range(3):
    print(f"Run {i+1}")
    print(zero_shot(sample_prompt)[0])
    print("-"*50)

Run 1
**Prime Number Checker Function**

Here's a simple Python function to check if a number is prime:

```python
def is_prime(n):
    """
    Checks if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """
    if n <= 1:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True
```

**Explanation**
---------------

This function works by checking if the input number `n` has any divisors other than 1 and itself. If it does, then `n` is not prime.

Here's a step-by-step breakdown:

1. If `n` is less than or equal to 1, it's not prime, so we return `False`.
2. We only need to check divisors up to the square root of `n`, because a larger factor of `n` would be a multiple of a smaller factor that has already been checked.
3. We iterate over the range from 2 to the square root of `n` (inclusive) and check if `n` is divisible by

In [ ]:
import json
import re

def generate_batch(n=10):
    prompt = f"""
    Generate {n} Python coding problems.

    Requirements:
    - Include difficulty: Easy, Medium, Hard
    - Balanced distribution
    - Output strictly valid JSON list only
    - No explanation text

    Format:
    [
      {{
        "instruction": "...",
        "difficulty": "Easy"
      }}
    ]
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    text = response.choices[0].message.content

    # Extract JSON safely
    match = re.search(r'\[.*\]', text, re.DOTALL)

    if match:
        return json.loads(match.group(0))
    else:
        return []

In [ ]:
dataset = []

for _ in range(10):  # 10 batches × 10 = 100
    batch = generate_batch(10)
    dataset.extend(batch)

print("Total samples:", len(dataset))

In [ ]:
with open("synthetic_code_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset saved.")

In [ ]:
df = pd.DataFrame(dataset)
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df["instruction"])
y = df["difficulty"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))